# Notebook 5 — Momentos, Matriz de Covariância e Encolhimento (Ledoit-Wolf)

Construção dos insumos da otimização de carteiras a partir dos **retornos saneados** (saída do
Notebook 4–5): o vetor de retornos esperados $\mu$, a matriz de covariância $\Sigma$ e a decisão
metodológica sobre o estimador de $\Sigma$.

## Decisões metodológicas

**Retornos simples para $\mu$ e $\Sigma$.** O modelo de média-variância (Markowitz) é um modelo
de período único cujo retorno de carteira é $R_p = \sum_i w_i R_i$ — uma identidade que só vale
para **retornos simples**. Portanto, tanto $\mu$ quanto $\Sigma$ são estimados sobre os retornos
simples saneados (os log-retornos seguem reservados à modelagem econométrica do NB11). A
anualização usa $\mu_{a} = \bar R\,\cdot 252$ e $\Sigma_{a} = \Sigma_d \cdot 252$.

**Por que o encolhimento (Ledoit-Wolf) importa — e onde.** Na amostra completa
($T\approx 3966$, $N=131$, $T/N\approx 30$) a covariância amostral é bem-condicionada. O problema
aparece na **estimação por janela móvel** que a otimização usa para rebalancear: numa janela de
252 pregões $T/N\approx 1{,}9$ (mal-condicionada), e numa de 126 pregões $T/N<1$, tornando
$\Sigma$ **singular** (autovalores nulos/negativos) e a inversão — necessária à fronteira
eficiente — numericamente instável. O estimador de **Ledoit & Wolf (2004)** encolhe a
covariância amostral $S$ em direção a um alvo bem-condicionado (identidade escalada),
$\Sigma^\* = \delta\,F + (1-\delta)\,S$, com a intensidade $\delta\in[0,1]$ estimada de forma
ótima a partir dos dados. Decisão: **$\Sigma$ amostral** para a descrição da amostra completa;
**Ledoit-Wolf** para a $\Sigma$ de janela móvel consumida pelos otimizadores (NB7+).

## 1. Configuração e carregamento dos retornos saneados

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

parent_path = Path.cwd().parent.parent
DIR_RETORNOS = parent_path / "data" / "Retornos"
OUTPUT_DIR   = parent_path / "data" / "Momentos"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TRADING_DAYS = 252

def _ler(nome):
    pq, csv = DIR_RETORNOS / f"{nome}.parquet", DIR_RETORNOS / f"{nome}.csv"
    if pq.exists():   return pd.read_parquet(pq)
    if csv.exists():  return pd.read_csv(csv, index_col=0, parse_dates=True)
    p = Path.cwd() / f"{nome}.csv"
    if p.exists():    return pd.read_csv(p, index_col=0, parse_dates=True)
    raise FileNotFoundError(f"{nome} não encontrado.")

ret = _ler("retornos_simples_saneado").sort_index()
ret.index = pd.to_datetime(ret.index)
rf = _ler("rf_diario").sort_index()
rf_diario_medio = rf["cdi_diario"].mean()

N, T = ret.shape[1], ret.shape[0]
print(f"Retornos saneados (simples): {T} pregões × {N} ativos | T/N = {T/N:.1f}")
print(f"r_f diário médio (CDI): {rf_diario_medio:.6f}  ({(1+rf_diario_medio)**TRADING_DAYS-1:.2%} a.a.)")

Retornos saneados (simples): 3966 pregões × 131 ativos | T/N = 30.3
r_f diário médio (CDI): 0.000369  (9.74% a.a.)


## 2. Momentos por ativo (anualizados)

In [2]:
mu      = ret.mean() * TRADING_DAYS
sigma   = ret.std()  * np.sqrt(TRADING_DAYS)
sharpe  = (ret.mean() - rf_diario_medio) / ret.std() * np.sqrt(TRADING_DAYS)

momentos = pd.DataFrame({
    "mu_anual":     mu,
    "sigma_anual":  sigma,
    "sharpe_anual": sharpe,
    "assimetria":   ret.skew(),
    "curtose_exc":  ret.kurtosis(),
})
momentos.index = [c.replace("ACAO_", "") for c in momentos.index]

print("Resumo dos momentos entre os ativos:")
print(momentos.describe().loc[["mean","min","max"]].T.round(4).to_string())
print("\nNota: μ por média histórica é ruidoso (e.g., ativos em distress têm μ muito negativo);")
print("o NB7 decide se usa μ diretamente, encolhe-o, ou adota carteiras que dispensam μ (mín-var).")
momentos.sort_values("sharpe_anual", ascending=False).head(8).round(4)

Resumo dos momentos entre os ativos:
                mean     min     max
mu_anual      0.0607 -1.1743  0.2888
sigma_anual   0.3993  0.2226  0.9502
sharpe_anual  0.0005 -1.8849  0.6035
assimetria    0.1873  0.0535  0.4854
curtose_exc   1.1414  0.6894  2.3787

Nota: μ por média histórica é ruidoso (e.g., ativos em distress têm μ muito negativo);
o NB7 decide se usa μ diretamente, encolhe-o, ou adota carteiras que dispensam μ (mín-var).


,mu_anual,sigma_anual,sharpe_anual,assimetria,curtose_exc
EQTL3,0.2450,0.2519,0.6035,0.1071,0.9771
WEGE3,0.2579,0.2876,0.5736,0.1502,0.8985
SAPR4,0.2663,0.3178,0.5454,0.2139,1.3955
UNIP6,0.2888,0.3718,0.5270,0.4019,2.1314
CSMG3,0.2591,0.3362,0.4941,0.0652,0.9845
SBSP3,0.2405,0.3216,0.4590,0.0890,0.9787
RADL3,0.2269,0.3083,0.4344,0.1056,0.6894
POMO3,0.2487,0.3668,0.4246,0.1700,1.2425


## 3. Covariância e correlação — amostra completa

In [3]:
def condicionamento(S):
    ev = np.linalg.eigvalsh(S)
    return ev.max() / max(ev.min(), 1e-18), ev.min()

Sigma_amostral = ret.cov() * TRADING_DAYS         # N×N anualizada
correlacao     = ret.corr()
kappa_full, emin_full = condicionamento(Sigma_amostral.values)

print(f"Σ amostral (completa): {Sigma_amostral.shape[0]}×{Sigma_amostral.shape[1]}")
print(f"  Número de condição κ = {kappa_full:,.0f}  | menor autovalor = {emin_full:.2e}")
print(f"  (T/N = {T/N:.0f} → bem-condicionada; inversão estável)")
print(f"\nCorrelação média (fora da diagonal): "
      f"{correlacao.values[np.triu_indices(N,1)].mean():.3f}")

Σ amostral (completa): 131×131
  Número de condição κ = 1,336  | menor autovalor = 3.66e-03
  (T/N = 30 → bem-condicionada; inversão estável)

Correlação média (fora da diagonal): 0.220


## 4. O problema de condicionamento na janela móvel

A otimização rebalanceia estimando $\Sigma$ em janelas. Abaixo, o número de condição cresce
(e a matriz fica singular) à medida que a janela encurta e $T/N$ cai abaixo de 1.

In [4]:
print("Janela | T/N  |        κ (amostral)        | menor autovalor")
print("-"*64)
for w in [504, 252, 126, 63]:
    Sw = ret.iloc[-w:].cov().values * TRADING_DAYS
    kw, eminw = condicionamento(Sw)
    sit = "SINGULAR" if w <= N else "ok"
    print(f"  {w:4d} | {w/N:4.2f} | {kw:>24,.0f} | {eminw:+.2e}  [{sit}]")
print("\n=> Em janelas <= N ativos, Σ amostral é singular: a fronteira eficiente exige encolhimento.")

Janela | T/N  |        κ (amostral)        | menor autovalor
----------------------------------------------------------------
   504 | 3.85 |                    3,103 | +1.05e-03  [ok]
   252 | 1.92 |                    7,589 | +4.19e-04  [ok]
   126 | 0.96 | 2,750,923,268,384,598,016 | -7.08e-17  [SINGULAR]
    63 | 0.48 | 3,161,117,981,028,030,976 | -4.33e-16  [SINGULAR]

=> Em janelas <= N ativos, Σ amostral é singular: a fronteira eficiente exige encolhimento.


## 5. Estimador de Ledoit & Wolf (2004) — encolhimento para identidade escalada

$\Sigma^\* = \delta F + (1-\delta) S$, com $F = \bar\mu\,I$ ($\bar\mu = \mathrm{tr}(S)/N$) e
$\delta = \beta^2/d^2$ estimado dos dados. Implementação direta (sem dependências externas);
confere com a referência `sklearn.covariance.ledoit_wolf` até precisão de máquina.

In [5]:
def ledoit_wolf(X):
    """X: T×N retornos (diários). Retorna (Sigma_shrunk_diaria, delta)."""
    X = np.asarray(X, float)
    T, N = X.shape
    Xc = X - X.mean(axis=0)
    S = (Xc.T @ Xc) / T                       # cov amostral (ddof=0)
    mu = np.trace(S) / N
    F = mu * np.eye(N)                         # alvo: identidade escalada
    d2 = np.sum((S - F) ** 2) / N              # ||S - F||_F^2 normalizada
    b2bar = 0.0
    for t in range(T):
        xt = np.outer(Xc[t], Xc[t])
        b2bar += np.sum((xt - S) ** 2) / N
    b2bar /= T ** 2
    b2 = min(b2bar, d2)
    delta = b2 / d2 if d2 > 0 else 0.0         # intensidade de encolhimento
    return delta * F + (1 - delta) * S, delta

def estimar_sigma(janela_retornos, metodo="ledoit_wolf", trading_days=TRADING_DAYS):
    """Helper para o NB7: Σ anualizada de uma janela de retornos simples."""
    if metodo == "amostral":
        return np.cov(janela_retornos, rowvar=False) * trading_days
    S, _ = ledoit_wolf(janela_retornos)
    return S * trading_days

# Demonstração: LW resgata as janelas mal-condicionadas/singulares
print("Janela | δ (encolhimento) | κ amostral  ->  κ Ledoit-Wolf")
print("-"*58)
for w in [252, 126, 63]:
    Xw = ret.iloc[-w:].values
    _, delta = ledoit_wolf(Xw)
    k_s, _ = condicionamento(np.cov(Xw, rowvar=False))
    k_lw, _ = condicionamento(estimar_sigma(Xw) / TRADING_DAYS)
    print(f"  {w:4d} |      {delta:.4f}      | {k_s:>10.2e}  ->  {k_lw:,.0f}")
print("\nQuanto menor a janela, maior δ (mais encolhimento) — e κ volta para a casa das dezenas.")

Janela | δ (encolhimento) | κ amostral  ->  κ Ledoit-Wolf
----------------------------------------------------------
   252 |      0.1150      |   7.59e+03  ->  172
   126 |      0.2232      |   1.09e+16  ->  77
    63 |      0.3567      |   1.25e+16  ->  43

Quanto menor a janela, maior δ (mais encolhimento) — e κ volta para a casa das dezenas.


## 6. Decisão e exportação dos insumos

- **Amostra completa (descritiva, NB6):** $\Sigma$ amostral — já bem-condicionada.
- **Janela móvel (otimização, NB7+):** $\Sigma$ por **Ledoit-Wolf**, via `estimar_sigma(...)`.

Exporta $\mu$, $\Sigma$ amostral, a correlação e a tabela de momentos. A função `ledoit_wolf`/
`estimar_sigma` deve ser reaproveitada no NB7 (idealmente extraída para um módulo `utils_cov.py`).

In [6]:
def _salvar(df, nome):
    df.to_csv(OUTPUT_DIR / f"{nome}.csv", float_format="%.10f")
    try:
        df.to_parquet(OUTPUT_DIR / f"{nome}.parquet", engine="pyarrow")
        print(f"  {nome}.csv + .parquet")
    except Exception as e:
        print(f"  {nome}.csv  [parquet: {e}]")

print("[+] Exportando insumos em:", OUTPUT_DIR, "\n")
_salvar(momentos, "momentos_anuais")
_salvar(mu.rename("mu_anual").to_frame(), "mu_anual")
_salvar(Sigma_amostral, "sigma_amostral_anual")
_salvar(correlacao, "correlacao")

# Σ Ledoit-Wolf da amostra completa (referência; full-sample quase não encolhe)
Sigma_lw_full, delta_full = ledoit_wolf(ret.values)
_salvar(pd.DataFrame(Sigma_lw_full * TRADING_DAYS, index=Sigma_amostral.index,
                     columns=Sigma_amostral.columns), "sigma_ledoitwolf_anual")

print(f"\n[i] δ na amostra completa = {delta_full:.4f} (encolhimento ínfimo, como esperado).")
print("\n" + "="*60)
print("NOTEBOOK 6 CONCLUÍDO")
print("="*60)
print(f"μ ({N}), Σ amostral ({N}×{N}), correlação e momentos exportados.")
print("Função estimar_sigma(janela, metodo='ledoit_wolf') pronta para o NB7.")
print("="*60)

[+] Exportando insumos em: c:\VSCodeWorkspace\TCC_Final\data\Momentos 

  momentos_anuais.csv + .parquet
  mu_anual.csv + .parquet
  sigma_amostral_anual.csv + .parquet
  correlacao.csv + .parquet
  sigma_ledoitwolf_anual.csv + .parquet

[i] δ na amostra completa = 0.0069 (encolhimento ínfimo, como esperado).

NOTEBOOK 6 CONCLUÍDO
μ (131), Σ amostral (131×131), correlação e momentos exportados.
Função estimar_sigma(janela, metodo='ledoit_wolf') pronta para o NB7.


## 7. Apêndice para o TCC — texto de redação

> *"Os insumos da otimização — o vetor de retornos esperados e a matriz de covariância — foram
> estimados sobre os retornos simples saneados, anualizados pelo fator de 252 pregões, em
> coerência com a aditividade dos retornos simples na agregação de carteira. Na amostra completa
> a covariância amostral mostrou-se bem-condicionada ($T/N\approx 30$). Contudo, na estimação por
> janela móvel utilizada no rebalanceamento, a razão $T/N$ aproxima-se ou cai abaixo da unidade,
> tornando a covariância amostral mal-condicionada ou singular e instável à inversão. Adotou-se,
> portanto, o estimador de encolhimento de Ledoit e Wolf (2004), que combina a covariância
> amostral com um alvo bem-condicionado (identidade escalada) segundo uma intensidade ótima
> estimada dos dados, restaurando a invertibilidade e reduzindo o número de condição em várias
> ordens de grandeza. Ressalva-se que o vetor de retornos esperados, estimado por média
> histórica, é sujeito a elevado erro amostral, questão tratada na etapa de otimização."*

**Artefatos em `data/Momentos/`:** `momentos_anuais.*`, `mu_anual.*`, `sigma_amostral_anual.*`,
`sigma_ledoitwolf_anual.*` e `correlacao.*`.